In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Бібліотеки завантажено')

Бібліотеки завантажено


## 1. Завантаження даних

In [2]:
df = pd.read_csv('../data/raw/supplier_level_features.csv')
print(f'Розмір датасету: {df.shape}')
df.head()

Розмір датасету: (3186, 12)


,supplier_id,num_bids,num_wins,avg_competitors,rejected_bids,complaints,log_avg_price_per_unit,log_avg_contract,contract_changes_share,win_rate,log_max_contract,log_min_contract
0,00034186,9,3,1.555556,0,0,11.968920,12.334259,0.0,0.333333,12.863367,11.642883
1,00135390,2,2,7.500000,0,0,19.622391,19.524181,0.0,1.000000,19.539096,19.509041
2,00236010,3,1,3.333333,0,0,13.979958,10.732061,0.0,0.333333,10.732061,10.732061
3,00445676,1,0,10.000000,0,0,15.789592,0.000000,0.0,0.000000,0.000000,0.000000
4,00445914,1,1,1.000000,0,0,11.685205,11.655433,0.0,1.000000,11.655433,11.655433


## 2. Загальна інформація про дані

In [3]:
print('Типи даних та кількість ненульових значень')
df.info()

Типи даних та кількість ненульових значень
<class 'pandas.DataFrame'>
RangeIndex: 3186 entries, 0 to 3185
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   supplier_id             3186 non-null   str    
 1   num_bids                3186 non-null   int64  
 2   num_wins                3186 non-null   int64  
 3   avg_competitors         3186 non-null   float64
 4   rejected_bids           3186 non-null   int64  
 5   complaints              3186 non-null   int64  
 6   log_avg_price_per_unit  3186 non-null   float64
 7   log_avg_contract        3186 non-null   float64
 8   contract_changes_share  3186 non-null   float64
 9   win_rate                3186 non-null   float64
 10  log_max_contract        3186 non-null   float64
 11  log_min_contract        3186 non-null   float64
dtypes: float64(7), int64(4), str(1)
memory usage: 298.8 KB


In [4]:
print('Пропущені значення')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'Пропущених значень немає!')

Пропущені значення
Пропущених значень немає!


In [5]:
print('Описова статистика')
df.describe()

Описова статистика


,num_bids,num_wins,avg_competitors,rejected_bids,complaints,log_avg_price_per_unit,log_avg_contract,contract_changes_share,win_rate,log_max_contract,log_min_contract
count,3186.000000,3186.000000,3186.000000,3186.0,3186.000000,3186.000000,3186.000000,3186.0,3186.000000,3186.000000,3186.000000
mean,2.247018,0.766164,2.573782,0.0,0.071249,13.000557,5.994898,0.0,0.388451,6.054193,5.848408
std,4.768506,1.668697,3.520873,0.0,0.713368,2.180346,6.406190,0.0,0.450021,6.471926,6.278135
min,1.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.0,0.000000,11.638081,0.000000,0.0,0.000000,0.000000,0.000000
50%,1.000000,0.000000,1.500000,0.0,0.000000,12.807860,0.000000,0.0,0.000000,0.000000,0.000000
75%,2.000000,1.000000,3.333333,0.0,0.000000,14.317595,12.309384,0.0,1.000000,12.427072,12.072700
max,78.000000,35.000000,26.000000,0.0,16.000000,21.321567,20.772056,0.0,1.000000,20.772056,20.772056


## 3. Аналіз цільової змінної (Target Distribution)

In [6]:
class_counts = df['Class'].value_counts()
fraud_pct = df['Class'].mean() * 100

print(f'Нормальних транзакцій:  {class_counts[0]:,} ({100 - fraud_pct:.3f}%)')
print(f'Шахрайських транзакцій: {class_counts[1]:,} ({fraud_pct:.3f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar plot
axes[0].bar(['Normal (0)', 'Fraud (1)'], class_counts.values,
            color=['steelblue', 'crimson'], edgecolor='black')
axes[0].set_title('Кількість транзакцій за класами', fontsize=14)
axes[0].set_ylabel('Кількість')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=['Normal', 'Fraud'],
            autopct='%1.3f%%', colors=['steelblue', 'crimson'],
            startangle=90, explode=(0, 0.1))
axes[1].set_title('Розподіл класів (%)', fontsize=14)

plt.tight_layout()
plt.savefig('../data/raw/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

KeyError: 'Class'

## 4. Матриця кореляції

In [ ]:
# Кореляція ознак з цільовою змінною
corr_with_target = df.corr()['Class'].drop('Class').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['crimson' if c > 0 else 'steelblue' for c in corr_with_target]
corr_with_target.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Кореляція ознак з цільовою змінною (Class)', fontsize=14)
ax.set_xlabel('Pearson Correlation')
ax.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('../data/raw/feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('Топ-5 позитивно корельованих ознак:')
print(corr_with_target.head())
print('\nТоп-5 негативно корельованих ознак:')
print(corr_with_target.tail())

In [ ]:
# Повна матриця кореляції (топ ознаки)
top_features = corr_with_target.abs().head(10).index.tolist() + ['Class']

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(df[top_features].corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True)
ax.set_title('Матриця кореляції (топ-10 ознак + Class)', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Висновки EDA

1. **Дисбаланс класів**: Шахрайські транзакції складають лише ~0.17% від усіх даних. Для моделювання необхідно використовувати відповідні метрики (F1, Precision, Recall, AUC-ROC), а не лише accuracy.

2. **Пропущені значення**: Датасет не містить пропущених значень.

3. **Ознаки V1-V28**: Анонімізовані за допомогою PCA. Найбільш корельовані з цільовою змінною: V17, V14, V12, V10 (негативна кореляція), V4, V11 (позитивна).

4. **Amount та Time**: Не анонімізовані. Шахрайські транзакції мають нижчу середню суму, але є викиди з великими сумами.

5. **Стратегія моделювання**: Використовувати RandomForest з підібраним порогом класифікації та метриками F1/AUC-ROC. Розглянути клас-ваги або oversampling (SMOTE).